## RDD

In [20]:
from pyspark import SparkConf, SparkContext

conf = (SparkConf()
        .setAppName("LR_5_Analyze_posts")
        .setMaster("local[*]")
)
sc = SparkContext(conf=conf)
path_file = "posts.csv"

rdd_posts = sc.textFile(path_file,)

- Исключите из RDD строку с заголовками (названиями столбцов)
- Напишите функцию парсинга , которая будет преобразовывать каждую строку в кортеж Python.

In [21]:
# если файл небольшой
# rdd_without_header = rdd_posts.filter(lambda x: x != rdd_posts.first())

# если файл большой, то используем zipWithIndex
rdd_without_header = rdd_posts.zipWithIndex().filter(lambda x: x[1] != 0).map(lambda x: x[0])

# проверяем, что получилось в итоге
print(rdd_without_header.take(5))

# функция парсинга строки
def parse_data (line):
    # разбиваем строку на части
    parts = line.split(",")
    post_id = int(parts[0].strip())
    post_type = parts[1].strip()
    comments = int(parts[2].strip())
    likes = int(parts[3].strip())
    return (post_id, post_type, comments, likes)

parsed_post_rdd = rdd_without_header.map(parse_data)
print(parsed_post_rdd.take(5))

['101,Carousel,268,16382', '102,Reel,138,9267', '103,Reel,1089,10100', '104,Reel,271,6943', '105,Reel,145,17158']
[(101, 'Carousel', 268, 16382), (102, 'Reel', 138, 9267), (103, 'Reel', 1089, 10100), (104, 'Reel', 271, 6943), (105, 'Reel', 145, 17158)]


### Общая статистика вовлеченности
- Посчитайте общее количество постов в датасете.
- Рассчитайте общее количество лайков по всем постам.
- Рассчитайте общее количество комментариев по всем постам.
- Определите среднее количество лайков на один пост. Округлите результат до одного знака после запятой.
- Определите среднее количество комментариев на один пост. Округлите результат до одного знака после запятой.

In [22]:
# Кол-во постов в датасете
total_posts = parsed_post_rdd.count()
print(f"Общее кол-во постов: {total_posts}")

Общее кол-во постов: 39


In [24]:
# Количество лайков по всем постам
total_likes = parsed_post_rdd.map(lambda x: x[3]).sum()
print(f"Общее количество лайков: {total_likes}")

Общее количество лайков: 707567


In [25]:
# Количество комментариев по всем постам
total_comments = parsed_post_rdd.map(lambda x: x[2]).sum()
print(f"Общее количество комментариев: {total_comments}")

Общее количество комментариев: 8387


In [28]:
# Среднее количество лайков на один пост
avg_likes_per_post = round(total_likes/total_posts, 1)
print(f"Среднее количество лайков на пост: {avg_likes_per_post}")

Среднее количество лайков на пост: 18142.7


In [29]:
# Среднее количество комментариев на один пост
avg_comments_per_post = round(total_comments/total_posts, 1)
print(f"Среднее количество комментариев на пост: {avg_comments_per_post}")

Среднее количество комментариев на пост: 215.1


### Анализ по типам постов
- Посчитайте количество постов каждого типа.
- Для каждого типа поста рассчитайте среднее количество лайков.  Округлите результат до одного знака после запятой.
- Для каждого типа поста рассчитайте среднее количество комментариев.  Округлите результат до одного знака после запятой.

In [32]:
# Количество постов каждого типа
post_types_cnt = parsed_post_rdd.map(lambda x: (x[1], 1)) \
                                .reduceByKey(lambda a, b: a + b) \
                                .collect()
print("Количество постов каждого типа:")
for post_type, count in sorted(post_types_cnt):
    print(f"    {post_type}: {count} постов")

Количество постов каждого типа:
    Carousel: 14 постов
    Image: 11 постов
    Reel: 14 постов


In [34]:
# Для каждого типа поста рассчитайте среднее количество лайков
# Для каждого типа поста рассчитайте среднее количество лайков
average_likes_by_type = parsed_post_rdd.map(lambda x: (x[1], (x[3], 1))) \
                                        .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                                        .mapValues(lambda x: round(x[0] / x[1] ,1)) \
                                        .collect()
print("Среднее количество лайков по типам постов:")
for post_type, avg_likes in sorted(average_likes_by_type):
    print(f"    {post_type}: {avg_likes}")

Среднее количество лайков по типам постов:
    Carousel: 20814.3
    Image: 18543.2
    Reel: 15156.6


In [35]:
# Для каждого типа поста рассчитайте среднее количество комментариев
average_comments_by_type = parsed_post_rdd.map(lambda x: (x[1], (x[2], 1))) \
                                           .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                                           .mapValues(lambda x: round(x[0] / x[1] ,1) if x[1] > 0 else 0) \
                                           .collect()
print("Среднее количество комментариев по типам постов:")
for post_type, avg_comments in sorted(average_comments_by_type):
    print(f"    {post_type}: {avg_comments}")

Среднее количество комментариев по типам постов:
    Carousel: 174.3
    Image: 137.5
    Reel: 316.7


### Вовлеченность
- Найдите Топ-5 постов с наибольшим количеством лайков. Выведите их Post_id, Post_Type , likes.
- Найдите Топ-5 постов с наибольшим количеством комментариев. Выведите их Post_id, Post_Type, comments

In [37]:
# Топ-5 постов с наибольшим количеством лайков
top_5_most_liked_posts = parsed_post_rdd.top(5, key=lambda x: x[3])
print("Топ-5 постов с наибольшим количеством лайков:")
for post_id, post_type, comments, likes in top_5_most_liked_posts:
    print(f"    Post ID: {post_id}, Тип: {post_type}, Лайки: {likes}")

# Топ-5 постов с наибольшим количеством комментариев
top_5_most_commented_posts = parsed_post_rdd.top(5, key=lambda x: x[2])
print("Топ-5 постов с наибольшим количеством комментариев:")
for post_id, post_type, comments, likes in top_5_most_commented_posts:
    print(f"    Post ID: {post_id}, Тип: {post_type}, Комментарии: {comments}")

Топ-5 постов с наибольшим количеством лайков:
    Post ID: 125, Тип: Carousel, Лайки: 79000
    Post ID: 131, Тип: Carousel, Лайки: 59716
    Post ID: 133, Тип: Carousel, Лайки: 58485
    Post ID: 132, Тип: Image, Лайки: 53254
    Post ID: 122, Тип: Image, Лайки: 50523
Топ-5 постов с наибольшим количеством комментариев:
    Post ID: 103, Тип: Reel, Комментарии: 1089
    Post ID: 109, Тип: Reel, Комментарии: 884
    Post ID: 123, Тип: Reel, Комментарии: 555
    Post ID: 131, Тип: Carousel, Комментарии: 507
    Post ID: 125, Тип: Carousel, Комментарии: 466


In [38]:
sc.stop()

## DataFrame

In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, sum, min, max, length, avg, round, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder \
        .appName("LR_5_Analyze_posts") \
        .master("local[*]") \
        .config("spark.sql.shuffle.partitions", 4) \
        .getOrCreate()

path_file = "posts.csv"

data_schema = StructType([
    StructField("Post_id", IntegerType(), True),
    StructField("Post_type", StringType(), True),
    StructField("comments", IntegerType(), True),
    StructField("likes", IntegerType(), True)
])

df_posts = spark.read.csv(
    path_file,
    header = True,
    schema=data_schema    
)
df_posts.show(10)

+-------+---------+--------+-----+
|Post_id|Post_type|comments|likes|
+-------+---------+--------+-----+
|    101| Carousel|     268|16382|
|    102|     Reel|     138| 9267|
|    103|     Reel|    1089|10100|
|    104|     Reel|     271| 6943|
|    105|     Reel|     145|17158|
|    106|     Reel|     143| 9683|
|    104|     Reel|     271| 6943|
|    107|     Reel|     132| 4287|
|    108|     Reel|     128| 7484|
|    109|     Reel|     884|48528|
+-------+---------+--------+-----+
only showing top 10 rows



### Общая статистика вовлеченности
- Посчитайте общее количество постов в датасете.
- Рассчитайте общее количество лайков по всем постам.
- Рассчитайте общее количество комментариев по всем постам.
- Определите среднее количество лайков на один пост. Округлите результат до одного знака после запятой.
- Определите среднее количество комментариев на один пост. Округлите результат до одного знака после запятой.

In [50]:
import builtins

print(f"Общее кол-во постов в датасете: {df_posts.count()}")

# Общее количество лайков по всем постам
total_likes = df_posts.agg(
    sum(col("likes")).alias("total_likes")
).collect()[0][0]
print(f"Общее количество лайков по всем постам: {total_likes}")

# общее количество комментариев по всем постам
total_comments = df_posts.agg(
    sum(col("comments")).alias("total_comments")
).collect()[0][0]
print(f"Общее количество комментариев по всем постам: {total_comments}")

# среднее количество лайков на один пост
avg_likes_per_post = builtins.round(total_likes/total_posts, 1)
print(f"Среднее количество лайков на один пост: {avg_likes_per_post}")

# среднее количество комментариев на один пост
avg_comments_per_post = builtins.round(total_comments/total_posts, 1)
print(f"Среднее количество комментариев на один пост: {avg_comments_per_post}")

Общее кол-во постов в датасете: 39
Общее количество лайков по всем постам: 707567
Общее количество комментариев по всем постам: 8387
Среднее количество лайков на один пост: 18142.7
Среднее количество комментариев на один пост: 215.1


### Анализ по типам постов
- Посчитайте количество постов каждого типа.
- Для каждого типа поста рассчитайте среднее количество лайков.  Округлите результат до одного знака после запятой.
- Для каждого типа поста рассчитайте среднее количество комментариев.  Округлите результат до одного знака после запятой.

In [51]:
# подсчет кол-во постов каждого типа
cnt_type_posts = df_posts.groupBy(col("Post_type")).agg(
    count("*").alias("cnt_post_type")
).orderBy(col("cnt_post_type").desc())

print("Кол-во постов каждого типа")
cnt_type_posts.show()


Кол-во постов каждого типа
+---------+-------------+
|Post_type|cnt_post_type|
+---------+-------------+
| Carousel|           14|
|     Reel|           14|
|    Image|           11|
+---------+-------------+



In [52]:
# Подсчет среднего кол-во лайков для каждого типа поста
avg_likes_per_type_post = df_posts.groupBy(col("Post_type")).agg(
    round(avg(col("likes")), 1).alias("avg_likes")
).orderBy(col("avg_likes").desc())

print("Среднее кол-во лайков для каждого типа поста: ")
avg_likes_per_type_post.show()

Среднее кол-во лайков для каждого типа поста: 
+---------+---------+
|Post_type|avg_likes|
+---------+---------+
| Carousel|  20814.3|
|    Image|  18543.2|
|     Reel|  15156.6|
+---------+---------+



In [53]:
# среднее количество комментариев для каждого типа поста
avg_comments_per_type_post = df_posts.groupBy(col("Post_type")).agg(
    round(avg(col("comments")), 1).alias("avg_comments")
).orderBy(col("avg_comments").desc())

print("Среднее кол-во комментариев для каждого типа поста: ")
avg_comments_per_type_post.show()

Среднее кол-во комментариев для каждого типа поста: 
+---------+------------+
|Post_type|avg_comments|
+---------+------------+
|     Reel|       316.7|
| Carousel|       174.3|
|    Image|       137.5|
+---------+------------+



### Вовлеченность
- Найдите Топ-5 постов с наибольшим количеством лайков. Выведите их Post_id, Post_Type , likes.
- Найдите Топ-5 постов с наибольшим количеством комментариев. Выведите их Post_id, Post_Type, comments

In [57]:
# Топ 5 по кол-ву лайков
top_5_likes = df_posts.orderBy(col("likes").desc()) \
            .limit(5) \
            .select(col("Post_id"), col("Post_type"), col("likes"))
print("ТОП 5 постов по кол-ву лайков")
top_5_likes.show()

ТОП 5 постов по кол-ву лайков
+-------+---------+-----+
|Post_id|Post_type|likes|
+-------+---------+-----+
|    125| Carousel|79000|
|    131| Carousel|59716|
|    133| Carousel|58485|
|    132|    Image|53254|
|    122|    Image|50523|
+-------+---------+-----+



In [58]:
# Топ 5 по кол-ву комментариев
top_5_comments = df_posts.orderBy(col("comments").desc()) \
                .limit(5) \
                .select(col("Post_id"), col("Post_type"), col("comments"))
print("ТОП 5 постов по кол-ву комментариев")
top_5_comments.show()

ТОП 5 постов по кол-ву комментариев
+-------+---------+--------+
|Post_id|Post_type|comments|
+-------+---------+--------+
|    103|     Reel|    1089|
|    109|     Reel|     884|
|    123|     Reel|     555|
|    131| Carousel|     507|
|    125| Carousel|     466|
+-------+---------+--------+



In [59]:
spark.stop()